Question 1

a)

In [1]:
import pandas as pd
import numpy as np

In [2]:
file_name = "lines.csv"

In [3]:
try:
    D = np.genfromtxt(file_name, delimiter=",", skip_header=1)
    
    # 2. Extract the data corresponding to the first line (x1 and y1)
    x1 = D[:, 0]
    y1 = D[:, 3]
    
    # 3. Calculate the centroids
    x_mean = np.mean(x1)
    y_mean = np.mean(y1)
    
    # 4. Center the data
    X_centered = x1 - x_mean
    Y_centered = y1 - y_mean
    
    # 5. Construct matrix A and perform SVD
    A = np.vstack((X_centered, Y_centered)).T
    U, S, Vt = np.linalg.svd(A)
    
    # 6. The normal vector (a, b) is the last row of V-transpose
    a, b = Vt[-1, :]
    
    # 7. Calculate final slope (m) and intercept (c)
    m = -a / b
    c = y_mean - m * x_mean

    print("=== Total Least Squares Fit (Line 1) ===")
    print(f"Slope (m): {m:.4f}")
    print(f"Y-Intercept (c): {c:.4f}")
    print(f"Line Equation: y = {m:.4f}x + {c:.4f}")

except FileNotFoundError:
    print(f"Error: Could not find '{file_name}'. Please ensure the file is in the same directory as this script.")

=== Total Least Squares Fit (Line 1) ===
Slope (m): 1.2207
Y-Intercept (c): -5.9872
Line Equation: y = 1.2207x + -5.9872


b)

In [4]:
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

In [5]:
points = np.column_stack((X_all, Y_all))

In [6]:
def tls_fit(pts):
    x, y = pts[:, 0], pts[:, 1]
    x_mean, y_mean = np.mean(x), np.mean(y)

    # Center the data and perform SVD
    A = np.vstack((x - x_mean, y - y_mean)).T
    _, _, Vt = np.linalg.svd(A)
    a, b = Vt[-1, :]

    # Return line parameters (a, b, c) for ax + by + c = 0
    c = -(a * x_mean + b * y_mean)
    return a, b, c

In [7]:
def find_single_line_ransac(pts, iterations=1000, threshold=0.5):
    best_inliers = []

    for _ in range(iterations):
        sample_idx = np.random.choice(len(pts), 2, replace=False)
        p1, p2 = pts[sample_idx]

        line = np.cross([p1[0], p1[1], 1], [p2[0], p2[1], 1])
        a, b, c = line

        norm = np.hypot(a, b)
        if norm == 0:
            continue
        a, b, c = a/norm, b/norm, c/norm

        distances = np.abs(a * pts[:, 0] + b * pts[:, 1] + c)

        inlier_idx = np.where(distances < threshold)[0]

        if len(inlier_idx) > len(best_inliers):
            best_inliers = inlier_idx

    if len(best_inliers) > 0:
        inlier_pts = pts[best_inliers]
        refined_model = tls_fit(inlier_pts)
        return refined_model, best_inliers

    return None, []

In [8]:
remaining_points = points.copy()
lines_found = []

distance_threshold = 0.5

In [9]:
for i in range(3):
    print(f"\n--- Searching for Line {i+1} ---")

    model, inlier_idx = find_single_line_ransac(
        remaining_points,
        iterations=2000,
        threshold=distance_threshold
    )

    if model is None:
        print("Not enough points to find another line.")
        break

    a, b, c = model
    m = -a / b
    c_int = -c / b

    lines_found.append((m, c_int))
    print(f"Result: y = {m:.4f}x + {c_int:.4f}")
    print(f"Inliers found: {len(inlier_idx)}")

    remaining_points = np.delete(remaining_points, inlier_idx, axis=0)
    print(f"Points remaining for next search: {len(remaining_points)}")


--- Searching for Line 1 ---
Result: y = -0.5075x + 2.1751
Inliers found: 84
Points remaining for next search: 216

--- Searching for Line 2 ---
Result: y = 1.0319x + 1.0806
Inliers found: 68
Points remaining for next search: 148

--- Searching for Line 3 ---
Result: y = 1.2851x + -5.9915
Inliers found: 65
Points remaining for next search: 83
